# Accepted Loan XGBoost Modeling And Tuning

This notebook trains and tunes `xgboost` candidates using the chronological baseline preprocessing exports; candidate search may use a stratified train-period sample for runtime. Thresholds are selected on validation only.


## 1. Setup

In [1]:
from __future__ import annotations

MODEL_FAMILY = 'xgboost'
MODEL_LABEL = 'XGBoost'


import json
import os
import time
from pathlib import Path

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_sample_weight

RANDOM_STATE = 42
TARGET_PRECISION = 0.40
REVIEW_RATES = [0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Cleaning").exists() and (candidate / "Modeling").exists():
            return candidate
    raise FileNotFoundError("Could not find CreditRiskRAG project root")

PROJECT_ROOT = find_project_root()
PREPROCESSING_DATASET_DIR = PROJECT_ROOT / "Modeling" / "Preprocessing" / "preprocessing_outputs" / "datasets"
MODELING_OUTPUT_ROOT = PROJECT_ROOT / "Modeling" / "modeling_outputs"
MODEL_OUTPUT_ROOT = MODELING_OUTPUT_ROOT / MODEL_FAMILY
TABLE_DIR = MODEL_OUTPUT_ROOT / "tables"
PLOT_DIR = MODEL_OUTPUT_ROOT / "plots"
MODEL_DIR = MODEL_OUTPUT_ROOT / "models"
for directory in [TABLE_DIR, PLOT_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Preprocessing datasets:", PREPROCESSING_DATASET_DIR)
print("Model outputs:", MODEL_OUTPUT_ROOT)

from xgboost import XGBClassifier


Project root: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG
Preprocessing datasets: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/Preprocessing/preprocessing_outputs/datasets
Model outputs: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost


## 2. Load Preprocessed Baseline Data

In [2]:

def save_table(df: pd.DataFrame, name: str) -> Path:
    path = TABLE_DIR / f"{MODEL_FAMILY}_{name}.csv"
    df.to_csv(path, index=False)
    print("Saved:", path)
    return path

def save_plot(fig, name: str) -> Path:
    path = PLOT_DIR / f"{MODEL_FAMILY}_{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", path)
    return path

def load_parquet(name: str) -> pd.DataFrame:
    path = PREPROCESSING_DATASET_DIR / f"{name}.parquet"
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_parquet(path)

X_train = load_parquet("baseline_train_X")
X_validation = load_parquet("baseline_validation_X")
X_test = load_parquet("baseline_test_X")
y_train = load_parquet("train_y")["target_bad"].astype(int)
y_validation = load_parquet("validation_y")["target_bad"].astype(int)
y_test = load_parquet("test_y")["target_bad"].astype(int)

input_summary = pd.DataFrame([
    {"split": "train", "rows": len(X_train), "columns": X_train.shape[1], "bad_rate": y_train.mean()},
    {"split": "validation", "rows": len(X_validation), "columns": X_validation.shape[1], "bad_rate": y_validation.mean()},
    {"split": "test", "rows": len(X_test), "columns": X_test.shape[1], "bad_rate": y_test.mean()},
])
input_summary["bad_rate"] = input_summary["bad_rate"].round(6)
save_table(input_summary, "input_summary")
display(input_summary)


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/tables/xgboost_input_summary.csv


,split,rows,columns,bad_rate
0,train,962641,100,0.188300
1,validation,186920,100,0.246763
2,test,195749,100,0.210315


## 3. Evaluation Helpers

In [3]:

def predict_positive_probability(model, X: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "n_jobs"):
        try:
            model.n_jobs = 1
        except Exception:
            pass
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    score = model.decision_function(X)
    return 1.0 / (1.0 + np.exp(-score))

def threshold_for_best_f1(y_true: pd.Series, y_score: np.ndarray) -> tuple[float, float, float, float]:
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    if len(thresholds) == 0:
        return 0.5, 0.0, 0.0, 0.0
    f1 = (2 * precision[:-1] * recall[:-1]) / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    idx = int(np.nanargmax(f1))
    return float(thresholds[idx]), float(f1[idx]), float(precision[idx]), float(recall[idx])

def threshold_for_target_precision(y_true: pd.Series, y_score: np.ndarray, target_precision: float) -> tuple[float, float, float, float]:
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    if len(thresholds) == 0:
        return 0.5, 0.0, 0.0, 0.0
    candidate = np.where(precision[:-1] >= target_precision)[0]
    if len(candidate) == 0:
        idx = int(np.nanargmax(precision[:-1]))
    else:
        idx = int(candidate[np.nanargmax(recall[:-1][candidate])])
    f1 = (2 * precision[idx] * recall[idx]) / max(precision[idx] + recall[idx], 1e-12)
    return float(thresholds[idx]), float(f1), float(precision[idx]), float(recall[idx])

def evaluate_at_threshold(model_name: str, split: str, y_true: pd.Series, y_score: np.ndarray, threshold: float, operating_point: str) -> dict:
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model_family": MODEL_FAMILY,
        "model": model_name,
        "split": split,
        "operating_point": operating_point,
        "rows": len(y_true),
        "bad_rate": round(float(y_true.mean()), 6),
        "threshold": round(float(threshold), 6),
        "roc_auc": round(float(roc_auc_score(y_true, y_score)), 6),
        "pr_auc": round(float(average_precision_score(y_true, y_score)), 6),
        "brier_score": round(float(brier_score_loss(y_true, y_score)), 6),
        "precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 6),
        "recall": round(float(recall_score(y_true, y_pred, zero_division=0)), 6),
        "f1": round(float(f1_score(y_true, y_pred, zero_division=0)), 6),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

def review_volume_metrics(model_name: str, split: str, y_true: pd.Series, y_score: np.ndarray) -> pd.DataFrame:
    order = np.argsort(-y_score)
    y_sorted = np.asarray(y_true)[order]
    base_bad_rate = float(np.mean(y_sorted))
    total_bad = int(y_sorted.sum())
    rows = []
    for rate in REVIEW_RATES:
        review_count = max(1, int(np.ceil(len(y_sorted) * rate)))
        reviewed = y_sorted[:review_count]
        captured_bad = int(reviewed.sum())
        precision = captured_bad / review_count
        recall = captured_bad / total_bad if total_bad else np.nan
        rows.append({
            "model_family": MODEL_FAMILY,
            "model": model_name,
            "split": split,
            "review_pct": round(rate * 100, 2),
            "review_count": int(review_count),
            "captured_bad": captured_bad,
            "precision": round(float(precision), 6),
            "recall": round(float(recall), 6),
            "base_bad_rate": round(base_bad_rate, 6),
            "lift_over_base_bad_rate": round(float(precision / base_bad_rate), 6) if base_bad_rate else np.nan,
        })
    return pd.DataFrame(rows)

def fit_candidates(candidates: list[dict], sample_rows: int | None = None) -> tuple[pd.DataFrame, dict]:
    if sample_rows and len(X_train) > sample_rows:
        sample_idx = y_train.groupby(y_train).sample(frac=sample_rows / len(y_train), random_state=RANDOM_STATE).index
        X_fit = X_train.loc[sample_idx]
        y_fit = y_train.loc[sample_idx]
    else:
        X_fit = X_train
        y_fit = y_train

    fitted_models = {}
    rows = []
    for candidate in candidates:
        name = candidate["candidate"]
        params = candidate["params"]
        print(f"Training {name}: {params}")
        start = time.perf_counter()
        model = build_model(params)
        fit_kwargs = build_fit_kwargs(y_fit)
        model.fit(X_fit, y_fit, **fit_kwargs)
        seconds = time.perf_counter() - start

        validation_score = predict_positive_probability(model, X_validation)
        best_threshold, best_f1, best_precision, best_recall = threshold_for_best_f1(y_validation, validation_score)
        precision_threshold, precision_f1, precision_value, precision_recall = threshold_for_target_precision(
            y_validation, validation_score, TARGET_PRECISION
        )
        row = {
            "model_family": MODEL_FAMILY,
            "candidate": name,
            "params": json.dumps(params, sort_keys=True),
            "fit_rows": len(X_fit),
            "fit_bad_rate": round(float(y_fit.mean()), 6),
            "fit_seconds": round(float(seconds), 3),
            "roc_auc": round(float(roc_auc_score(y_validation, validation_score)), 6),
            "pr_auc": round(float(average_precision_score(y_validation, validation_score)), 6),
            "best_f1_threshold": round(best_threshold, 6),
            "best_f1": round(best_f1, 6),
            "best_f1_precision": round(best_precision, 6),
            "best_f1_recall": round(best_recall, 6),
            "target_precision_threshold": round(precision_threshold, 6),
            "target_precision_f1": round(precision_f1, 6),
            "target_precision": round(precision_value, 6),
            "target_precision_recall": round(precision_recall, 6),
        }
        rows.append(row)
        fitted_models[name] = model
        print(f"Finished {name}: best_f1={best_f1:.4f}, precision={best_precision:.4f}, recall={best_recall:.4f}, seconds={seconds:.1f}")
    results = pd.DataFrame(rows).sort_values(["best_f1", "best_f1_precision", "pr_auc"], ascending=False)
    return results, fitted_models

def evaluate_selected_model(candidate_row: pd.Series, model) -> pd.DataFrame:
    scores = {
        "train": predict_positive_probability(model, X_train),
        "validation": predict_positive_probability(model, X_validation),
        "test": predict_positive_probability(model, X_test),
    }
    y_parts = {"train": y_train, "validation": y_validation, "test": y_test}
    rows = []
    for split, y_part in y_parts.items():
        rows.append(evaluate_at_threshold(candidate_row.candidate, split, y_part, scores[split], candidate_row.best_f1_threshold, "best_validation_f1"))
        rows.append(evaluate_at_threshold(candidate_row.candidate, split, y_part, scores[split], candidate_row.target_precision_threshold, "target_validation_precision"))
    review_rows = [review_volume_metrics(candidate_row.candidate, split, y_parts[split], scores[split]) for split in ["validation", "test"]]
    review_df = pd.concat(review_rows, ignore_index=True)
    return pd.DataFrame(rows), review_df


## 4. Candidate Grid

In [4]:
def build_model(params: dict) -> XGBClassifier:
    return XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        max_depth=params["max_depth"],
        min_child_weight=params["min_child_weight"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        reg_lambda=params["reg_lambda"],
        reg_alpha=params["reg_alpha"],
        scale_pos_weight=float((y_train == 0).sum() / max((y_train == 1).sum(), 1)),
        random_state=RANDOM_STATE,
        n_jobs=1,
    )

def build_fit_kwargs(y_fit: pd.Series) -> dict:
    return {}

CANDIDATES = [
    {"candidate": "xgboost_01", "params": {"n_estimators": 220, "learning_rate": 0.04, "max_depth": 4, "min_child_weight": 8, "subsample": 0.85, "colsample_bytree": 0.85, "reg_lambda": 2.0, "reg_alpha": 0.0}},
    {"candidate": "xgboost_02", "params": {"n_estimators": 260, "learning_rate": 0.035, "max_depth": 4, "min_child_weight": 12, "subsample": 0.85, "colsample_bytree": 0.80, "reg_lambda": 3.0, "reg_alpha": 0.1}},
    {"candidate": "xgboost_03", "params": {"n_estimators": 300, "learning_rate": 0.03, "max_depth": 5, "min_child_weight": 10, "subsample": 0.80, "colsample_bytree": 0.80, "reg_lambda": 4.0, "reg_alpha": 0.1}},
    {"candidate": "xgboost_04", "params": {"n_estimators": 240, "learning_rate": 0.045, "max_depth": 5, "min_child_weight": 16, "subsample": 0.90, "colsample_bytree": 0.85, "reg_lambda": 3.0, "reg_alpha": 0.2}},
    {"candidate": "xgboost_05", "params": {"n_estimators": 350, "learning_rate": 0.025, "max_depth": 6, "min_child_weight": 16, "subsample": 0.80, "colsample_bytree": 0.75, "reg_lambda": 5.0, "reg_alpha": 0.2}},
    {"candidate": "xgboost_06", "params": {"n_estimators": 180, "learning_rate": 0.06, "max_depth": 3, "min_child_weight": 8, "subsample": 0.90, "colsample_bytree": 0.90, "reg_lambda": 2.0, "reg_alpha": 0.0}},
]
FIT_SAMPLE_ROWS = 300_000


## 5. Train, Tune, And Evaluate

In [5]:

candidate_results, fitted_models = fit_candidates(CANDIDATES, sample_rows=FIT_SAMPLE_ROWS)
save_table(candidate_results, "candidate_results")
display(candidate_results)

winner = candidate_results.iloc[0]
selected_model = fitted_models[winner.candidate]
selected_metrics, review_volume_precision = evaluate_selected_model(winner, selected_model)
save_table(pd.DataFrame([winner]), "selected_candidate")
save_table(selected_metrics, "selected_model_metrics")
save_table(review_volume_precision, "review_volume_precision")
display(selected_metrics)
display(review_volume_precision)

model_path = MODEL_DIR / f"{MODEL_FAMILY}_selected_model.joblib"
joblib.dump(selected_model, model_path)
artifact_table = pd.DataFrame([{
    "model_family": MODEL_FAMILY,
    "candidate": winner.candidate,
    "artifact_path": str(model_path),
}])
save_table(artifact_table, "model_artifact")
print("Saved:", model_path)

for metric in ["f1", "precision", "recall", "pr_auc", "roc_auc"]:
    plot_df = selected_metrics[(selected_metrics["split"].isin(["validation", "test"])) & (selected_metrics["operating_point"] == "best_validation_f1")]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(plot_df["split"], plot_df[metric])
    ax.set_title(f"{MODEL_LABEL} {metric.upper()} by split")
    ax.set_ylim(0, max(plot_df[metric].max() * 1.15, 0.05))
    ax.grid(axis="y", alpha=0.25)
    save_plot(fig, f"selected_{metric}_comparison")

fig, ax = plt.subplots(figsize=(8, 4.5))
for split, group in review_volume_precision.groupby("split"):
    ax.plot(group["review_pct"], group["precision"], marker="o", label=split)
ax.set_title(f"{MODEL_LABEL} precision at fixed review volumes")
ax.set_xlabel("Reviewed applications (%)")
ax.set_ylabel("Precision")
ax.grid(alpha=0.25)
ax.legend()
save_plot(fig, "precision_by_review_volume")


Training xgboost_01: {'n_estimators': 220, 'learning_rate': 0.04, 'max_depth': 4, 'min_child_weight': 8, 'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_lambda': 2.0, 'reg_alpha': 0.0}


Finished xgboost_01: best_f1=0.4721, precision=0.3508, recall=0.7216, seconds=6.2
Training xgboost_02: {'n_estimators': 260, 'learning_rate': 0.035, 'max_depth': 4, 'min_child_weight': 12, 'subsample': 0.85, 'colsample_bytree': 0.8, 'reg_lambda': 3.0, 'reg_alpha': 0.1}


Finished xgboost_02: best_f1=0.4723, precision=0.3541, recall=0.7089, seconds=7.4
Training xgboost_03: {'n_estimators': 300, 'learning_rate': 0.03, 'max_depth': 5, 'min_child_weight': 10, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 4.0, 'reg_alpha': 0.1}


Finished xgboost_03: best_f1=0.4735, precision=0.3593, recall=0.6941, seconds=8.9
Training xgboost_04: {'n_estimators': 240, 'learning_rate': 0.045, 'max_depth': 5, 'min_child_weight': 16, 'subsample': 0.9, 'colsample_bytree': 0.85, 'reg_lambda': 3.0, 'reg_alpha': 0.2}


Finished xgboost_04: best_f1=0.4738, precision=0.3557, recall=0.7092, seconds=7.0
Training xgboost_05: {'n_estimators': 350, 'learning_rate': 0.025, 'max_depth': 6, 'min_child_weight': 16, 'subsample': 0.8, 'colsample_bytree': 0.75, 'reg_lambda': 5.0, 'reg_alpha': 0.2}


...

,model_family,model,split,operating_point,rows,bad_rate,threshold,roc_auc,pr_auc,brier_score,precision,recall,f1,tn,fp,fn,tp
0,xgboost,xgboost_05,train,best_validation_f1,962641,0.188300,0.470698,0.734468,0.395261,0.207900,0.303218,0.726605,0.427878,478716,302660,49557,131708
1,xgboost,xgboost_05,train,target_validation_precision,962641,0.188300,0.546067,0.734468,0.395261,0.207900,0.347097,0.587747,0.436448,580974,200402,74727,106538
2,xgboost,xgboost_05,validation,best_validation_f1,186920,0.246763,0.470698,0.701999,0.425400,0.217751,0.357988,0.700748,0.473884,82829,57966,13803,32322
3,xgboost,xgboost_05,validation,target_validation_precision,186920,0.246763,0.546067,0.701999,0.425400,0.217751,0.400003,0.546667,0.461974,102973,37822,20910,25215
4,xgboost,xgboost_05,test,best_validation_f1,195749,0.210315,0.470698,0.710193,0.379590,0.212441,0.318981,0.701061,0.438462,92960,61620,12307,28862
5,xgboost,xgboost_05,test,target_validation_precision,195749,0.210315,0.546067,0.710193,0.379590,0.212441,0.356460,0.553353,0.433602,113452,41128,18388,22781


,model_family,model,split,review_pct,review_count,captured_bad,precision,recall,base_bad_rate,lift_over_base_bad_rate
0,xgboost,xgboost_05,validation,1.0,1870,1208,0.645989,0.026190,0.246763,2.617850
1,xgboost,xgboost_05,validation,2.0,3739,2297,0.614335,0.049799,0.246763,2.489573
2,xgboost,xgboost_05,validation,5.0,9346,5221,0.558635,0.113192,0.246763,2.263848
3,xgboost,xgboost_05,validation,10.0,18692,9587,0.512893,0.207848,0.246763,2.078482
4,xgboost,xgboost_05,validation,15.0,28038,13331,0.475462,0.289019,0.246763,1.926793
5,xgboost,xgboost_05,validation,20.0,37384,16835,0.450326,0.364986,0.246763,1.824932
6,xgboost,xgboost_05,validation,25.0,46730,20043,0.428911,0.434537,0.246763,1.738146
7,xgboost,xgboost_05,validation,30.0,56076,23057,0.411174,0.499881,0.246763,1.666269
8,xgboost,xgboost_05,test,1.0,1958,1125,0.574566,0.027326,0.210315,2.731927
9,xgboost,xgboost_05,test,2.0,3915,2123,0.542273,0.051568,0.210315,2.578383


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/tables/xgboost_model_artifact.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/models/xgboost_selected_model.joblib
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/plots/xgboost_selected_f1_comparison.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/plots/xgboost_selected_precision_comparison.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/plots/xgboost_selected_recall_comparison.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Fin

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/plots/xgboost_precision_by_review_volume.png


PosixPath('/Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/plots/xgboost_precision_by_review_volume.png')

## 6. Confusion Matrix And Per-Class Metrics

Show the confusion-matrix layout for each split and operating point. Class `0` is `Fully Paid`; class `1` is `Charged Off`. Precision, recall, and F1 are also reported separately for each class.

In [6]:
def build_confusion_matrix_tables(metrics: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    matrix_rows = []
    per_class_rows = []

    def safe_div(num: float, den: float) -> float:
        return float(num / den) if den else 0.0

    for _, row in metrics.iterrows():
        tn = int(row["tn"])
        fp = int(row["fp"])
        fn = int(row["fn"])
        tp = int(row["tp"])
        base = {
            "model_family": MODEL_FAMILY,
            "candidate": row["model"],
            "split": row["split"],
            "operating_point": row["operating_point"],
            "threshold": row["threshold"],
        }

        matrix_rows.extend([
            {**base, "actual_label": 0, "actual_class": "Fully Paid", "predicted_label": 0, "predicted_class": "Fully Paid", "count": tn, "cell": "TN"},
            {**base, "actual_label": 0, "actual_class": "Fully Paid", "predicted_label": 1, "predicted_class": "Charged Off", "count": fp, "cell": "FP"},
            {**base, "actual_label": 1, "actual_class": "Charged Off", "predicted_label": 0, "predicted_class": "Fully Paid", "count": fn, "cell": "FN"},
            {**base, "actual_label": 1, "actual_class": "Charged Off", "predicted_label": 1, "predicted_class": "Charged Off", "count": tp, "cell": "TP"},
        ])

        precision_0 = safe_div(tn, tn + fn)
        recall_0 = safe_div(tn, tn + fp)
        f1_0 = safe_div(2 * precision_0 * recall_0, precision_0 + recall_0)
        precision_1 = safe_div(tp, tp + fp)
        recall_1 = safe_div(tp, tp + fn)
        f1_1 = safe_div(2 * precision_1 * recall_1, precision_1 + recall_1)

        per_class_rows.extend([
            {**base, "class_label": 0, "class_name": "Fully Paid", "precision": round(precision_0, 6), "recall": round(recall_0, 6), "f1": round(f1_0, 6), "support": tn + fp},
            {**base, "class_label": 1, "class_name": "Charged Off", "precision": round(precision_1, 6), "recall": round(recall_1, 6), "f1": round(f1_1, 6), "support": tp + fn},
        ])

    return pd.DataFrame(matrix_rows), pd.DataFrame(per_class_rows)


if "selected_metrics" not in globals():
    selected_metrics_path = TABLE_DIR / f"{MODEL_FAMILY}_selected_model_metrics.csv"
    if not selected_metrics_path.exists():
        raise FileNotFoundError(f"Missing {selected_metrics_path}. Run the training/evaluation section first.")
    selected_metrics = pd.read_csv(selected_metrics_path)

confusion_matrix_long, per_class_metrics = build_confusion_matrix_tables(selected_metrics)
save_table(confusion_matrix_long, "confusion_matrix")
save_table(per_class_metrics, "per_class_metrics")

best_f1_confusion_matrix = confusion_matrix_long[
    (confusion_matrix_long["split"].isin(["validation", "test"]))
    & (confusion_matrix_long["operating_point"] == "best_validation_f1")
]
best_f1_per_class_metrics = per_class_metrics[
    (per_class_metrics["split"].isin(["validation", "test"]))
    & (per_class_metrics["operating_point"] == "best_validation_f1")
]

display(best_f1_confusion_matrix)
display(best_f1_per_class_metrics)

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/tables/xgboost_confusion_matrix.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/tables/xgboost_per_class_metrics.csv


,model_family,candidate,split,operating_point,threshold,actual_label,actual_class,predicted_label,predicted_class,count,cell
8,xgboost,xgboost_05,validation,best_validation_f1,0.470698,0,Fully Paid,0,Fully Paid,82829,TN
9,xgboost,xgboost_05,validation,best_validation_f1,0.470698,0,Fully Paid,1,Charged Off,57966,FP
10,xgboost,xgboost_05,validation,best_validation_f1,0.470698,1,Charged Off,0,Fully Paid,13803,FN
11,xgboost,xgboost_05,validation,best_validation_f1,0.470698,1,Charged Off,1,Charged Off,32322,TP
16,xgboost,xgboost_05,test,best_validation_f1,0.470698,0,Fully Paid,0,Fully Paid,92960,TN
17,xgboost,xgboost_05,test,best_validation_f1,0.470698,0,Fully Paid,1,Charged Off,61620,FP
18,xgboost,xgboost_05,test,best_validation_f1,0.470698,1,Charged Off,0,Fully Paid,12307,FN
19,xgboost,xgboost_05,test,best_validation_f1,0.470698,1,Charged Off,1,Charged Off,28862,TP


,model_family,candidate,split,operating_point,threshold,class_label,class_name,precision,recall,f1,support
4,xgboost,xgboost_05,validation,best_validation_f1,0.470698,0,Fully Paid,0.857159,0.588295,0.697722,140795
5,xgboost,xgboost_05,validation,best_validation_f1,0.470698,1,Charged Off,0.357988,0.700748,0.473884,46125
8,xgboost,xgboost_05,test,best_validation_f1,0.470698,0,Fully Paid,0.883088,0.601371,0.715498,154580
9,xgboost,xgboost_05,test,best_validation_f1,0.470698,1,Charged Off,0.318981,0.701061,0.438462,41169


## 7. No-Grade/Subgrade Modeling

Train and tune the same candidate grid on the `baseline_no_grade_subgrade` feature set so the advanced model families can be compared against the prior no-grade/subgrade baseline outputs.


In [7]:
NO_GRADE_SUFFIX = "no_grade_subgrade"
CENTRAL_NO_GRADE_METRICS_PATH = MODELING_OUTPUT_ROOT / "tables" / "no_grade_subgrade_model_metrics.csv"

def save_or_replace_central_no_grade_metrics(new_metrics: pd.DataFrame) -> Path:
    CENTRAL_NO_GRADE_METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
    if CENTRAL_NO_GRADE_METRICS_PATH.exists():
        existing = pd.read_csv(CENTRAL_NO_GRADE_METRICS_PATH)
        existing = existing[~existing["model"].str.startswith(f"{MODEL_FAMILY}_{NO_GRADE_SUFFIX}")]
        combined = pd.concat([existing, new_metrics], ignore_index=True)
    else:
        combined = new_metrics.copy()
    combined.to_csv(CENTRAL_NO_GRADE_METRICS_PATH, index=False)
    print("Saved:", CENTRAL_NO_GRADE_METRICS_PATH)
    return CENTRAL_NO_GRADE_METRICS_PATH

def run_no_grade_subgrade_modeling() -> None:
    global X_train, X_validation, X_test

    original_X_train = X_train
    original_X_validation = X_validation
    original_X_test = X_test

    try:
        X_train = load_parquet("baseline_no_grade_subgrade_train_X")
        X_validation = load_parquet("baseline_no_grade_subgrade_validation_X")
        X_test = load_parquet("baseline_no_grade_subgrade_test_X")

        no_grade_input_summary = pd.DataFrame([
            {"split": "train", "rows": len(X_train), "columns": X_train.shape[1], "bad_rate": y_train.mean()},
            {"split": "validation", "rows": len(X_validation), "columns": X_validation.shape[1], "bad_rate": y_validation.mean()},
            {"split": "test", "rows": len(X_test), "columns": X_test.shape[1], "bad_rate": y_test.mean()},
        ])
        no_grade_input_summary["bad_rate"] = no_grade_input_summary["bad_rate"].round(6)
        save_table(no_grade_input_summary, f"{NO_GRADE_SUFFIX}_input_summary")
        display(no_grade_input_summary)

        no_grade_candidates = [
            {**candidate, "candidate": f"{candidate['candidate']}_{NO_GRADE_SUFFIX}"}
            for candidate in CANDIDATES
        ]
        no_grade_candidate_results, no_grade_fitted_models = fit_candidates(
            no_grade_candidates,
            sample_rows=FIT_SAMPLE_ROWS,
        )
        save_table(no_grade_candidate_results, f"{NO_GRADE_SUFFIX}_candidate_results")
        display(no_grade_candidate_results)

        no_grade_winner = no_grade_candidate_results.iloc[0]
        no_grade_selected_model = no_grade_fitted_models[no_grade_winner.candidate]
        no_grade_selected_metrics, no_grade_review_volume_precision = evaluate_selected_model(
            no_grade_winner,
            no_grade_selected_model,
        )

        no_grade_selected_metrics = no_grade_selected_metrics[
            no_grade_selected_metrics["operating_point"] == "best_validation_f1"
        ].copy()
        no_grade_selected_metrics = no_grade_selected_metrics.drop(columns=["model_family", "operating_point"])

        save_table(pd.DataFrame([no_grade_winner]), f"{NO_GRADE_SUFFIX}_selected_candidate")
        save_table(no_grade_selected_metrics, f"{NO_GRADE_SUFFIX}_selected_model_metrics")
        save_table(no_grade_review_volume_precision, f"{NO_GRADE_SUFFIX}_review_volume_precision")
        save_or_replace_central_no_grade_metrics(no_grade_selected_metrics)
        display(no_grade_selected_metrics)
        display(no_grade_review_volume_precision)

        no_grade_model_path = MODEL_DIR / f"{MODEL_FAMILY}_{NO_GRADE_SUFFIX}_selected_model.joblib"
        joblib.dump(no_grade_selected_model, no_grade_model_path)
        no_grade_artifact_table = pd.DataFrame([{
            "model_family": MODEL_FAMILY,
            "candidate": no_grade_winner.candidate,
            "feature_set": "baseline_no_grade_subgrade",
            "artifact_path": str(no_grade_model_path),
        }])
        save_table(no_grade_artifact_table, f"{NO_GRADE_SUFFIX}_model_artifact")
        print("Saved:", no_grade_model_path)

        no_grade_confusion_matrix_long, no_grade_per_class_metrics = build_confusion_matrix_tables(
            no_grade_selected_metrics.assign(
                model_family=MODEL_FAMILY,
                operating_point="best_validation_f1",
            )
        )
        save_table(no_grade_confusion_matrix_long, f"{NO_GRADE_SUFFIX}_confusion_matrix")
        save_table(no_grade_per_class_metrics, f"{NO_GRADE_SUFFIX}_per_class_metrics")
        display(no_grade_confusion_matrix_long)
        display(no_grade_per_class_metrics)

        for metric in ["f1", "precision", "recall", "pr_auc", "roc_auc"]:
            plot_df = no_grade_selected_metrics[no_grade_selected_metrics["split"].isin(["validation", "test"])]
            fig, ax = plt.subplots(figsize=(7, 4))
            ax.bar(plot_df["split"], plot_df[metric])
            ax.set_title(f"{MODEL_LABEL} no-grade/subgrade {metric.upper()} by split")
            ax.set_ylim(0, max(plot_df[metric].max() * 1.15, 0.05))
            ax.grid(axis="y", alpha=0.25)
            save_plot(fig, f"{NO_GRADE_SUFFIX}_selected_{metric}_comparison")
    finally:
        X_train = original_X_train
        X_validation = original_X_validation
        X_test = original_X_test

run_no_grade_subgrade_modeling()


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/tables/xgboost_no_grade_subgrade_input_summary.csv


,split,rows,columns,bad_rate
0,train,962641,58,0.188300
1,validation,186920,58,0.246763
2,test,195749,58,0.210315


Training xgboost_01_no_grade_subgrade: {'n_estimators': 220, 'learning_rate': 0.04, 'max_depth': 4, 'min_child_weight': 8, 'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_lambda': 2.0, 'reg_alpha': 0.0}


Finished xgboost_01_no_grade_subgrade: best_f1=0.4724, precision=0.3605, recall=0.6850, seconds=4.0
Training xgboost_02_no_grade_subgrade: {'n_estimators': 260, 'learning_rate': 0.035, 'max_depth': 4, 'min_child_weight': 12, 'subsample': 0.85, 'colsample_bytree': 0.8, 'reg_lambda': 3.0, 'reg_alpha': 0.1}


Finished xgboost_02_no_grade_subgrade: best_f1=0.4725, precision=0.3567, recall=0.6998, seconds=4.7
Training xgboost_03_no_grade_subgrade: {'n_estimators': 300, 'learning_rate': 0.03, 'max_depth': 5, 'min_child_weight': 10, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 4.0, 'reg_alpha': 0.1}


...

,model_family,model,split,review_pct,review_count,captured_bad,precision,recall,base_bad_rate,lift_over_base_bad_rate
0,xgboost,xgboost_05_no_grade_subgrade,validation,1.0,1870,1213,0.648663,0.026298,0.246763,2.628685
1,xgboost,xgboost_05_no_grade_subgrade,validation,2.0,3739,2287,0.611661,0.049583,0.246763,2.478735
2,xgboost,xgboost_05_no_grade_subgrade,validation,5.0,9346,5226,0.559170,0.113301,0.246763,2.266016
3,xgboost,xgboost_05_no_grade_subgrade,validation,10.0,18692,9603,0.513749,0.208195,0.246763,2.081951
4,xgboost,xgboost_05_no_grade_subgrade,validation,15.0,28038,13333,0.475533,0.289062,0.246763,1.927082
5,xgboost,xgboost_05_no_grade_subgrade,validation,20.0,37384,16778,0.448802,0.363751,0.246763,1.818753
6,xgboost,xgboost_05_no_grade_subgrade,validation,25.0,46730,20027,0.428568,0.434190,0.246763,1.736759
7,xgboost,xgboost_05_no_grade_subgrade,validation,30.0,56076,23089,0.411745,0.500575,0.246763,1.668582
8,xgboost,xgboost_05_no_grade_subgrade,test,1.0,1958,1103,0.563330,0.026792,0.210315,2.678503
9,xgboost,xgboost_05_no_grade_subgrade,test,2.0,3915,2147,0.548404,0.052151,0.210315,2.607531


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/tables/xgboost_no_grade_subgrade_model_artifact.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/models/xgboost_no_grade_subgrade_selected_model.joblib
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/tables/xgboost_no_grade_subgrade_confusion_matrix.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/tables/xgboost_no_grade_subgrade_per_class_metrics.csv


,model_family,candidate,split,operating_point,threshold,actual_label,actual_class,predicted_label,predicted_class,count,cell
0,xgboost,xgboost_05_no_grade_subgrade,train,best_validation_f1,0.462322,0,Fully Paid,0,Fully Paid,466525,TN
1,xgboost,xgboost_05_no_grade_subgrade,train,best_validation_f1,0.462322,0,Fully Paid,1,Charged Off,314851,FP
2,xgboost,xgboost_05_no_grade_subgrade,train,best_validation_f1,0.462322,1,Charged Off,0,Fully Paid,47450,FN
3,xgboost,xgboost_05_no_grade_subgrade,train,best_validation_f1,0.462322,1,Charged Off,1,Charged Off,133815,TP
4,xgboost,xgboost_05_no_grade_subgrade,validation,best_validation_f1,0.462322,0,Fully Paid,0,Fully Paid,80813,TN
5,xgboost,xgboost_05_no_grade_subgrade,validation,best_validation_f1,0.462322,0,Fully Paid,1,Charged Off,59982,FP
6,xgboost,xgboost_05_no_grade_subgrade,validation,best_validation_f1,0.462322,1,Charged Off,0,Fully Paid,13162,FN
7,xgboost,xgboost_05_no_grade_subgrade,validation,best_validation_f1,0.462322,1,Charged Off,1,Charged Off,32963,TP
8,xgboost,xgboost_05_no_grade_subgrade,test,best_validation_f1,0.462322,0,Fully Paid,0,Fully Paid,90021,TN
9,xgboost,xgboost_05_no_grade_subgrade,test,best_validation_f1,0.462322,0,Fully Paid,1,Charged Off,64559,FP


,model_family,candidate,split,operating_point,threshold,class_label,class_name,precision,recall,f1,support
0,xgboost,xgboost_05_no_grade_subgrade,train,best_validation_f1,0.462322,0,Fully Paid,0.907680,0.597056,0.720307,781376
1,xgboost,xgboost_05_no_grade_subgrade,train,best_validation_f1,0.462322,1,Charged Off,0.298251,0.738229,0.424856,181265
2,xgboost,xgboost_05_no_grade_subgrade,validation,best_validation_f1,0.462322,0,Fully Paid,0.859941,0.573976,0.688444,140795
3,xgboost,xgboost_05_no_grade_subgrade,validation,best_validation_f1,0.462322,1,Charged Off,0.354651,0.714645,0.474049,46125
4,xgboost,xgboost_05_no_grade_subgrade,test,best_validation_f1,0.462322,0,Fully Paid,0.886095,0.582359,0.702814,154580
5,xgboost,xgboost_05_no_grade_subgrade,test,best_validation_f1,0.462322,1,Charged Off,0.314340,0.718915,0.437421,41169


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/plots/xgboost_no_grade_subgrade_selected_f1_comparison.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/plots/xgboost_no_grade_subgrade_selected_precision_comparison.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/plots/xgboost_no_grade_subgrade_selected_recall_comparison.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/plots/xgboost_no_grade_subgrade_selected_pr_auc_comparison.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/xgboost/plots/xgboost_no_grade_subgrade_selected_roc

## 8. Notes

Use validation metrics for model/threshold selection. Use test metrics only for final reporting after selection.
